# Script: Raw_Data_Preprocessing
- written by Jasmin L. Walter (jawalter@uos.de)
- reads in nested json files and returns flattened csv files
- does not change anything in the data, only extracts all variables from all 9 layers of the nested json files and saves them in data frames / csv files unchanged

In [1]:
import os
import json
import numpy as np
import re
import pandas as pd
#import networkx as nx
import glob
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from timeit import default_timer as timer
import time

# Customize to run scripts - paths, subject ids to run etc.

In [2]:
os.getcwd()

'/home/sand94/projects/VReGTrET/GraphTheory_ET_VR_Westbrueck/Pre-processing'

In [3]:
DATA_PATH = '/mnt/f/big-data/vr_data/Data/raw_amsterdam/'

PROCESSED_DATA_PATH = '/mnt/f/big-data/vr_data/Data/preprocessed/'

# Getting the Folder without hidden files in ascending order 
DATA_FOLDER = sorted([f for f in os.listdir(DATA_PATH) if not f.startswith('.')], key=str.lower)
PROCESSED_DATA_FOLDER = sorted([f for f in os.listdir(PROCESSED_DATA_PATH) if not f.startswith('.')], key=str.lower)


In [4]:
subIDs = []
for sub in DATA_FOLDER:
    if sub[0:4].isdigit():
        subIDs.append(sub[0:4])
    else:
        pass
subIDs = np.unique(subIDs)
print(subIDs)

['4003' '4004' '4005' '4006' '4007' '4008' '4009' '4010' '4011' '4012'
 '4013' '4014' '4015' '4016' '4017' '4018' '4019' '4020' '4021' '4022'
 '4023' '4024' '4025' '4026' '4028' '4029' '4031' '4034']


# Main part - flatten all nested data structures and save as csv

In [5]:
# if no ray cast information is available, the data frame will be filled with nan values
# create empty data frames with nan values and correct variable names
columns1 = ['hitObjectColliderName_1','ordinalOfHit_1','hitPointOnObject.x_1','hitPointOnObject.y_1','hitPointOnObject.z_1',
            'hitObjectColliderBoundsCenter.x_1','hitObjectColliderBoundsCenter.y_1','hitObjectColliderBoundsCenter.z_1']

columns2 = ['hitObjectColliderName_2','ordinalOfHit_2','hitPointOnObject.x_2','hitPointOnObject.y_2','hitPointOnObject.z_2',
            'hitObjectColliderBoundsCenter.x_2','hitObjectColliderBoundsCenter.y_2','hitObjectColliderBoundsCenter.z_2']

columnsRCall = ['hitObjectColliderName_1','ordinalOfHit_1','hitPointOnObject.x_1','hitPointOnObject.y_1','hitPointOnObject.z_1',
                'hitObjectColliderBoundsCenter.x_1','hitObjectColliderBoundsCenter.y_1','hitObjectColliderBoundsCenter.z_1',
                'hitObjectColliderName_2','ordinalOfHit_2','hitPointOnObject.x_2','hitPointOnObject.y_2','hitPointOnObject.z_2',
                'hitObjectColliderBoundsCenter.x_2','hitObjectColliderBoundsCenter.y_2','hitObjectColliderBoundsCenter.z_2',
                'DataRow']

emptyDF1 = pd.DataFrame(np.nan,index=[0], columns= columns1)
emptyDF2 = pd.DataFrame(np.nan,index=[0], columns= columns2)



#########################################################################################################
# data loop through all subjects and sessions

subcount = 0


for subject in subIDs:
    
    subcount +=1
    print('Subject ' 
          + str(subject) 
          + ' started - ' 
          + str(subcount) 
          + '/' 
          + str(len(subIDs)) 
          + ' subjects')
    
#     # Create empty dataframe for later concatenation
# complete_exploration_df = pd.DataFrame(columns = col_names)
#     complete_exploration_df.head()
    
    
    # change dir into the subject folder 
    CURRENT_SUBJECT_FOLDER = sorted([f for f in os.listdir(DATA_PATH+str(subject)) if not f.startswith('.')], key=str.lower)
    # get the data files according to the subject, ignoring OnQuit files
    subject_files = sorted([f for f in CURRENT_SUBJECT_FOLDER 
                             if f.startswith(str(subject)+'_Expl_S_') and f.endswith("OnQuit.json") == False], 
                            key=str.lower) 
    
    # the following works as long as the data name format is as follows:
    # 'subjectID'_Expl_S_'SessionNumber'_ET_'EyeTrackingSessionNumber'_'UnixTimestamp'.json
    folder_files = list()
    
    # loop through the subject folder and save all numbers
    for file in subject_files:
        folder_files.append(re.findall(r'\d+', file))
    
    # Extract all SubIDs (only one), SessionNumbers, ET_SessionNumbers (and Timestamps)
    try:
        SubID, SessionNumbers, ET_SessionNumbers, UnixTimestamp1, UnixTimeStamp2 = map(list, zip(*folder_files))
    except:
        print('\tSubject ' 
              + str(subject)
              + ' Filename is not valid!')
        
#     print(SubID)
#     print(SessionNumbers)
#     print(ET_SessionNumbers)
#     print(UnixTimestamp1)
#     print(UnixTimeStamp2)
    
    session_number = int(max(SessionNumbers)) # the maximum session number of the particular subject
    ET_session_number = int(max(ET_SessionNumbers)) # the maximum ET session number of the particular subject
    
    
    # print info of how many files were found 
    
    print(len(SubID), ' files were found for participant ', SubID[0])
    print('A maximum of ', session_number, 'sessions were found and will be processed')
        
# --------- second layer - exploration session loop ---------

    # loop over exploration sessions
    for EXP_session in range(session_number+1):

        # extract the exploration data files for each session - but exclude OnQuit files
        subject_data = sorted([f for f in CURRENT_SUBJECT_FOLDER if f.startswith(str(subject) + '_Expl_S_' + str(EXP_session)) 
                               and f.endswith("OnQuit.json") == False], key=str.lower)


        print("\tTotal Sessionfiles: "
              + str(len(subject_data))
              + " - Exploration Session "
              + str(EXP_session))

        ET_session_count = 0 # session count

# --------- third layer - eye tracking session loop ---------

        # loop over separate eye tracking sessions
        for fileName in subject_data:

            print('load data of file ', fileName)

            print('Path: ', DATA_PATH + str(subject) + '/' + fileName)
            # open the JSON file as dictionary
            with open(DATA_PATH + str(subject) + '/' + fileName) as datafile:
                try:
                    print("read file")
                    dataR = '['+ datafile.read()
                    dataR = dataR[:len(dataR)] + "]"
                except:
                    print("reading did not work")

                subject_session = json.loads(dataR)
                print("data loaded")
                print('time is: ', time.ctime())



##################################################################################################################

            # Data flattening part: 
            # first save the overall trial information


            infoDF = pd.json_normalize(subject_session[0]['trials'][0])
            infoDF = infoDF.drop(columns=['dataPoints'])
            infoDF.insert(0,'FileInfo',fileName[0:18])
            infoDF.to_csv(PROCESSED_DATA_PATH + fileName[0:18] + '_infoSummary.csv', index = False)
            print('trial info saved')
            

            # flatten the majority of the variables into currentDF data frame
            currentDF_raw = pd.json_normalize(subject_session[0]['trials'][0]['dataPoints'])

            # remove the 'rayCastHitsCombinedEyes' column as it still contains a nested data structure
            dataDF = currentDF_raw.drop(columns=['rayCastHitsCombinedEyes'])
            
            # create an empty data frame of the required size
            rayCastData_df = pd.DataFrame(np.nan,index=range(len(subject_session[0]['trials'][0]['dataPoints'])), columns= columnsRCall)

            # now loop through the individual trials and flatten the data
            for index in range(len(subject_session[0]['trials'][0]['dataPoints'])):
                
                # depending on the size of the ray cast data - flatten data and appand it to currentDF data frame
                # the variables are renamed to make the differentiation of first and second order collider hits more intuitive
                #lengthRCData = len(subject_session[0]['trials'][0]['dataPoints'][index]['rayCastHitsCombinedEyes'][0])
                lengthRCData = len(currentDF_raw.at[index,'rayCastHitsCombinedEyes'])
                
                
                if lengthRCData ==0: #case: no ray cast data is available = no collider was hit

                    combineDF = pd.concat([emptyDF1, emptyDF2], axis=1)
                    combineDF.insert(len(combineDF.columns), 'DataRow',index)


                elif lengthRCData == 1: # case: only one collider was hit, there is no secondary hit

                    pdRC1= pd.json_normalize(currentDF_raw.at[index,'rayCastHitsCombinedEyes'][0]).rename(
                        columns = {'hitObjectColliderName':'hitObjectColliderName_1',
                                   'ordinalOfHit':'ordinalOfHit_1',
                                   'hitPointOnObject.x':'hitPointOnObject.x_1',
                                   'hitPointOnObject.y':'hitPointOnObject.y_1',
                                   'hitPointOnObject.z':'hitPointOnObject.z_1',
                                   'hitObjectColliderBoundsCenter.x':'hitObjectColliderBoundsCenter.x_1',
                                   'hitObjectColliderBoundsCenter.y':'hitObjectColliderBoundsCenter.y_1',
                                   'hitObjectColliderBoundsCenter.z':'hitObjectColliderBoundsCenter.z_1'})
                    combineDF = pd.concat([pdRC1, emptyDF2], axis=1)
                    combineDF.insert(len(combineDF.columns), 'DataRow',index)

                elif lengthRCData == 2: # case: two collider were hit 

                    pdRC1= pd.json_normalize(currentDF_raw.at[index,'rayCastHitsCombinedEyes'][0]).rename(
                        columns = {'hitObjectColliderName':'hitObjectColliderName_1',
                                   'ordinalOfHit':'ordinalOfHit_1',
                                   'hitPointOnObject.x':'hitPointOnObject.x_1',
                                   'hitPointOnObject.y':'hitPointOnObject.y_1',
                                   'hitPointOnObject.z':'hitPointOnObject.z_1',
                                   'hitObjectColliderBoundsCenter.x':'hitObjectColliderBoundsCenter.x_1',
                                   'hitObjectColliderBoundsCenter.y':'hitObjectColliderBoundsCenter.y_1',
                                   'hitObjectColliderBoundsCenter.z':'hitObjectColliderBoundsCenter.z_1'})

                    pdRC2 = pd.json_normalize(currentDF_raw.at[index,'rayCastHitsCombinedEyes'][1]).rename(
                        columns = {'hitObjectColliderName':'hitObjectColliderName_2',
                                   'ordinalOfHit':'ordinalOfHit_2',
                                   'hitPointOnObject.x':'hitPointOnObject.x_2',
                                   'hitPointOnObject.y':'hitPointOnObject.y_2',
                                   'hitPointOnObject.z':'hitPointOnObject.z_2',
                                   'hitObjectColliderBoundsCenter.x':'hitObjectColliderBoundsCenter.x_2',
                                   'hitObjectColliderBoundsCenter.y':'hitObjectColliderBoundsCenter.y_2',
                                   'hitObjectColliderBoundsCenter.z':'hitObjectColliderBoundsCenter.z_2'})
                    combineDF = pd.concat([pdRC1, pdRC2], axis=1)
                    combineDF.insert(len(combineDF.columns), 'DataRow',index)


                else:
                    print('!!!an exception occured in the ray cast data flattening in trial ', index)

                # now add the new data row to the data overview
                # rayCastData_df = [rayCastData_df]

            
                rayCastData_df.loc[index] = combineDF.loc[0]
                
            flatData_df = pd.concat([dataDF,rayCastData_df],axis=1)   

            print('saving data')
            flatData_df.to_csv(PROCESSED_DATA_PATH + fileName[0:18] + '_flattened.csv', index = False)
            print('data saved')
            print('time is: ', time.ctime())


Subject 4003 started - 1/28 subjects
4  files were found for participant  4003
A maximum of  1 sessions were found and will be processed
	Total Sessionfiles: 0 - Exploration Session 0
	Total Sessionfiles: 4 - Exploration Session 1
load data of file  4003_Expl_S_1_ET_1_1732177385.17388.json
Path:  /mnt/f/big-data/vr_data/Data/raw_amsterdam/4003/4003_Expl_S_1_ET_1_1732177385.17388.json
read file
data loaded
time is:  Mon Feb 17 12:32:39 2025
trial info saved
saving data
data saved
time is:  Mon Feb 17 12:32:40 2025
load data of file  4003_Expl_S_1_ET_2_1732177998.34681.json
Path:  /mnt/f/big-data/vr_data/Data/raw_amsterdam/4003/4003_Expl_S_1_ET_2_1732177998.34681.json
read file
data loaded
time is:  Mon Feb 17 12:32:46 2025
trial info saved
saving data
data saved
time is:  Mon Feb 17 12:38:00 2025
load data of file  4003_Expl_S_1_ET_3_1732178950.8367.json
Path:  /mnt/f/big-data/vr_data/Data/raw_amsterdam/4003/4003_Expl_S_1_ET_3_1732178950.8367.json
read file
data loaded
time is:  Mon Feb